# Challenge 1 — Cartes d’identité des CSV

Exécuter toutes les cellules avec l’environnement **indusense** (pandas requis).
Le notebook fonctionne depuis la racine du projet ou depuis son dossier `notebooks`.
Les sources restent intactes ; les rapports sont écrits en UTF-8 dans
`artifacts/ingestion/identities/YYYYMMDD/`, à la date locale de Paris.
Le nom de sortie `releves_incidents` reprend volontairement la consigne.


In [1]:
import csv
import io
import math
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "datas").is_dir() and (ROOT.parent / "datas").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "datas").is_dir():
    raise FileNotFoundError("Exécuter depuis la racine du projet ou le dossier notebooks.")

PARIS = ZoneInfo("Europe/Paris")
TEMPORAL_FORMATS = {
    "DATE": "%Y-%m-%d",
    "TIME": "%H:%M",
    "DATETIME": "%Y-%m-%d %H:%M:%S",
}
# Contrat explicite pour ces sources : une valeur incorrecte ne doit pas
# transformer une colonne de mesures ou de dates en simple texte.
EXPECTED_TYPES = {
    "incident_id": "IDENTITY",
    "machine_id": "IDENTITY",
    "operator_badge": "IDENTITY",
    "date": "DATE",
    "time": "TIME",
    "timestamp": "DATETIME",
    "severity": "INTEGER",
    "pieces_produced": "INTEGER",
    "temperature_c": "FLOAT",
    "pressure_bar": "FLOAT",
    "voltage_mean_v": "FLOAT",
    "rotation_mean_rpm": "FLOAT",
}


## 1. Lecture et métadonnées

Décodage strict : UTF-8 (avec prise en charge du BOM), puis Windows-1252 et
Latin-1 en repli. Un encodage de repli est une hypothèse explicitement signalée,
car ces encodages ne sont pas toujours distinguables à partir des octets.
Les cellules vides ou composées d’espaces sont manquantes ; les autres chaînes
(par exemple un identifiant avec des zéros initiaux) sont préservées.


In [2]:
def read_csv_source(source):
    source = Path(source).resolve(strict=True)
    raw = source.read_bytes()
    notes = []
    for charset in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            text = raw.decode(charset, errors="strict")
            break
        except UnicodeDecodeError:
            continue
    if charset != "utf-8-sig":
        notes.append(
            f"UTF-8 invalide : lecture en {charset}, encodage de repli "
            "non déterminable avec certitude à partir des seuls octets."
        )
    charset_label = (
        "UTF-8 avec BOM" if raw.startswith(b"\xef\xbb\xbf") else "UTF-8"
    ) if charset == "utf-8-sig" else charset
    if not text.strip():
        raise ValueError(f"Fichier vide : {source}")
    try:
        separator = csv.Sniffer().sniff(text[:65536], delimiters=",;\t").delimiter
    except csv.Error as exc:
        raise ValueError(f"Séparateur indéterminé pour {source}") from exc
    frame = pd.read_csv(
        io.StringIO(text), sep=separator, dtype="string", keep_default_na=False,
        skip_blank_lines=False, on_bad_lines="error",
    )
    frame = frame.replace(r"^\s*$", pd.NA, regex=True)
    return frame, {
        "source": source, "size": len(raw), "charset": charset_label,
        "separator": separator, "notes": notes,
    }


## 2. Inférence et validation des types

Les identifiants sont reconnus avant les types numériques. Le contrat des
colonnes connues est vérifié sur les valeurs. Pour une colonne inconnue,
l’inférence exige que toutes les valeurs présentes respectent le format proposé.
Les valeurs invalides sont comptées et signalées, puis exclues des statistiques.


In [3]:
def infer_columns(frame):
    columns = {}
    issues = []
    temporal_patterns = {
        "DATE": r"\d{4}-\d{2}-\d{2}",
        "TIME": r"\d{2}:\d{2}",
        "DATETIME": r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}",
    }
    for name in frame.columns:
        series = frame[name]
        present = series.dropna()
        normalized = name.lower()
        kind = EXPECTED_TYPES.get(normalized)
        if kind is None and (normalized.endswith("_id") or normalized.endswith("_badge")):
            kind = "IDENTITY"
        if kind is None and normalized.startswith("type_"):
            kind = "INTEGER"
        if kind is None and not present.empty:
            for candidate, pattern in temporal_patterns.items():
                if present.str.fullmatch(pattern).all():
                    parsed = pd.to_datetime(
                        present, format=TEMPORAL_FORMATS[candidate], errors="coerce", exact=True
                    )
                    if parsed.notna().all():
                        kind = candidate
                        break
            if kind is None:
                numbers = pd.to_numeric(present, errors="coerce")
                if numbers.notna().all() and not numbers.isin([math.inf, -math.inf]).any():
                    kind = "INTEGER" if numbers.mod(1).eq(0).all() else "FLOAT"
        kind = kind or "TEXT"
        converted = series
        if kind in ("INTEGER", "FLOAT"):
            converted = pd.to_numeric(series, errors="coerce")
            converted = converted.mask(converted.isin([math.inf, -math.inf]))
            if kind == "INTEGER":
                converted = converted.mask(converted.mod(1).ne(0).fillna(False))
            if normalized.startswith("type_"):
                converted = converted.where(converted.isin([0, 1]))
        elif kind in TEMPORAL_FORMATS:
            converted = pd.to_datetime(
                series, format=TEMPORAL_FORMATS[kind], errors="coerce", exact=True
            )
            # Imposer le format annoncé, même si le parseur accepte une variante.
            converted = converted.where(
                series.str.fullmatch(temporal_patterns[kind]).fillna(False)
            )
        invalid = series.notna() & converted.isna()
        if invalid.any():
            examples = ", ".join(repr(value) for value in series[invalid].head(3))
            issues.append(
                f"{name} : {int(invalid.sum())} valeur(s) incompatible(s) avec {kind} "
                f"exclue(s) des statistiques. Exemples : {examples}"
            )
        columns[name] = {"type": kind, "values": converted}
    return columns, issues


In [4]:
def compute_statistics(columns):
    statistics = []
    for name, info in columns.items():
        kind = info["type"]
        if kind not in ("INTEGER", "FLOAT", "DATE", "TIME", "DATETIME"):
            continue
        values = info["values"].dropna()
        row = {
            "name": name, "type": kind, "min": None, "max": None,
            "mean": None, "median": None,
        }
        if not values.empty:
            row.update({"min": values.min(), "max": values.max()})
            if kind in ("INTEGER", "FLOAT"):
                row.update({"mean": values.mean(), "median": values.median()})
        statistics.append(row)
    return statistics


## 3. Rendu Markdown et génération

La fin du traitement est relevée après l’analyse, juste avant l’écriture.
La taille utilise les multiples de 1 024 octets (Ko, Mo, Go).
Une sortie existante est protégée : si deux exécutions commencent à la même
seconde, attendre une seconde avant de relancer.


In [5]:
def render_markdown(metadata, frame, columns, statistics, begin, end):
    def escape(value):
        return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

    def format_value(value, kind, aggregate=False):
        if value is None or pd.isna(value):
            return "N/A"
        if kind in TEMPORAL_FORMATS:
            return value.strftime(TEMPORAL_FORMATS[kind])
        if kind == "INTEGER" and not aggregate:
            return str(int(value))
        return format(float(value), ".12g")

    size = float(metadata["size"])
    unit = "o"
    for unit in ("o", "Ko", "Mo", "Go", "To"):
        if size < 1024 or unit == "To":
            break
        size /= 1024
    separator_label = "tabulation" if metadata["separator"] == "\t" else metadata["separator"]
    lines = [
        f"# Carte d’identité — {metadata['source'].name}", "",
        f"- Source: {metadata['source']}",
        f"- Filsize: {size:.2f} {unit}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {metadata['charset']}",
        f"- Rows Number: {len(frame)}",
        f"- Cols Number: {len(frame.columns)}",
        f"- Date: {begin:%Y-%m-%d} (Europe/Paris)",
        f"- Séparateur: {separator_label}", "",
        "## Column list", "", "| Colname | Infered Type |", "| --- | --- |",
    ]
    lines.extend(f"| {escape(name)} | {info['type']} |" for name, info in columns.items())
    lines.extend([
        "", "## Statistiques", "",
        "| Col name | Min value | Max value | Mean value | Median value |",
        "| --- | --- | --- | --- | --- |",
    ])
    for row in statistics:
        values = [escape(row["name"])] + [
            format_value(row[key], row["type"], key in ("mean", "median"))
            for key in ("min", "max", "mean", "median")
        ]
        lines.append("| " + " | ".join(values) + " |")
    lines.extend([
        "", "Les valeurs manquantes et les valeurs invalides signalées ci-dessous sont exclues "
        "des calculs. N/A indique une statistique non applicable ou sans valeur exploitable.",
        "Les identifiants sont conservés comme chaînes et exclus des statistiques numériques.",
        "", "## Observations", "",
    ])
    notes = metadata["notes"] or ["Aucune anomalie de conversion détectée."]
    lines.extend(f"- {escape(note)}" for note in notes)
    return "\n".join(lines) + "\n"


def generate_identity(source, output_stem, root=ROOT):
    begin = datetime.now(PARIS)
    frame, metadata = read_csv_source(source)
    columns, issues = infer_columns(frame)
    metadata["notes"].extend(issues)
    statistics = compute_statistics(columns)
    output_dir = root / "artifacts" / "ingestion" / "identities" / begin.strftime("%Y%m%d")
    output_dir.mkdir(parents=True, exist_ok=True)
    output = output_dir / f"{output_stem}_{begin:%H%M%S}.md"
    end = datetime.now(PARIS)
    report = render_markdown(metadata, frame, columns, statistics, begin, end)
    with output.open("x", encoding="utf-8") as stream:
        stream.write(report)
    return {
        "path": output, "rows": len(frame), "cols": len(frame.columns),
        "columns": columns, "statistics": statistics, "notes": metadata["notes"],
        "begin": begin, "end": end,
    }


In [ ]:
reports = [
    generate_identity(ROOT / "datas" / "releves_incidents.csv", "releves_incidents"),
    generate_identity(ROOT / "datas" / "telemetry.csv", "telemetry"),
]
for report in reports:
    print(f"{report['path']} — {report['rows']} lignes, {report['cols']} colonnes")
    for note in report["notes"]:
        print(f"  Observation : {note}")


/home/ybourdea/PycharmProjects/indusense/artifacts/ingestion/identities/20261120/releves_incidents_143453.md — 1245 lignes, 18 colonnes
/home/ybourdea/PycharmProjects/indusense/artifacts/ingestion/identities/20261120/telemetry_143453.md — 135626 lignes, 7 colonnes
